In [2]:

# ============================================================
# CODEALPHA AI INTERNSHIP
# TASK 2 - CHATBOT FOR FAQs
# ============================================================

# Install required libraries
!pip install -q gradio scikit-learn


# ============================================================
# IMPORT LIBRARIES
# ============================================================

import gradio as gr
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ============================================================
# FAQ DATASET
# ============================================================

faqs = [

    {
        "question": "What is CodeAlpha?",
        "answer": "CodeAlpha is a software development company that provides internship opportunities and practical experience in emerging technologies."
    },

    {
        "question": "What is the CodeAlpha internship?",
        "answer": "The CodeAlpha internship provides students with practical experience in areas such as artificial intelligence, software development and other emerging technologies."
    },

    {
        "question": "How many tasks should I complete?",
        "answer": "According to the internship instructions, participants need to complete a minimum of two tasks. Participants may complete two or three tasks from the listed tasks."
    },

    {
        "question": "How do I submit my project?",
        "answer": "You need to upload your complete source code to GitHub and submit your completed task using the CodeAlpha submission form."
    },

    {
        "question": "What should I name my GitHub repository?",
        "answer": "The required repository naming format is CodeAlpha_ProjectName."
    },

    {
        "question": "Do I need to upload my source code to GitHub?",
        "answer": "Yes. The internship instructions require participants to upload their complete source code to GitHub."
    },

    {
        "question": "Do I need to post my project on LinkedIn?",
        "answer": "Yes. The internship instructions ask participants to post a video explanation of their project on LinkedIn and include the GitHub repository link."
    },

    {
        "question": "Should I tag CodeAlpha on LinkedIn?",
        "answer": "Yes. The internship instructions ask participants to share their internship status on LinkedIn and tag CodeAlpha."
    },

    {
        "question": "What is artificial intelligence?",
        "answer": "Artificial Intelligence, or AI, is a field of computer science focused on creating systems that can perform tasks that normally require human intelligence."
    },

    {
        "question": "What is machine learning?",
        "answer": "Machine learning is a branch of artificial intelligence in which computer systems learn patterns from data to make predictions or decisions."
    },

    {
        "question": "What is natural language processing?",
        "answer": "Natural Language Processing, or NLP, is a field of artificial intelligence that focuses on enabling computers to process and understand human language."
    },

    {
        "question": "What is a chatbot?",
        "answer": "A chatbot is a computer program that interacts with users through text or natural language and provides responses to their questions."
    },

    {
        "question": "What is cosine similarity?",
        "answer": "Cosine similarity is a mathematical technique used to measure how similar two text vectors are. It is used in this project to find the FAQ that is most similar to the user's question."
    },

    {
        "question": "What is TF-IDF?",
        "answer": "TF-IDF stands for Term Frequency-Inverse Document Frequency. It is a technique that converts text into numerical vectors based on the importance of words."
    },

    {
        "question": "How does this chatbot work?",
        "answer": "The chatbot preprocesses the user's question, converts it into a TF-IDF vector, compares it with the FAQ vectors using cosine similarity, and returns the answer with the highest similarity."
    },

    {
        "question": "What technologies are used in this chatbot?",
        "answer": "This chatbot uses Python, Gradio and Scikit-learn. TF-IDF is used for text vectorization and cosine similarity is used for FAQ matching."
    },

    {
        "question": "What happens if the chatbot does not understand my question?",
        "answer": "If the similarity score is below the minimum threshold, the chatbot informs the user that it could not find a suitable answer."
    },

    {
        "question": "What is NLP preprocessing?",
        "answer": "NLP preprocessing prepares text for analysis by performing operations such as converting text to lowercase, removing unnecessary characters and normalizing whitespace."
    }

]


# ============================================================
# TEXT PREPROCESSING
# ============================================================

def preprocess_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove special characters
    text = re.sub(
        r"[^a-zA-Z0-9\s]",
        "",
        text
    )

    # Remove extra spaces
    text = re.sub(
        r"\s+",
        " ",
        text
    ).strip()

    return text


# ============================================================
# PREPARE FAQ QUESTIONS
# ============================================================

faq_questions = [
    preprocess_text(faq["question"])
    for faq in faqs
]


# ============================================================
# TF-IDF VECTORIZER
# ============================================================

vectorizer = TfidfVectorizer(
    ngram_range=(1, 2)
)


faq_vectors = vectorizer.fit_transform(
    faq_questions
)


# ============================================================
# FIND BEST ANSWER
# ============================================================

def get_answer(user_question):

    # Check empty question
    if not user_question or not user_question.strip():

        return (
            "🤖 Please enter a question."
        )


    # Preprocess user question
    cleaned_question = preprocess_text(
        user_question
    )


    # Convert question to TF-IDF vector
    user_vector = vectorizer.transform(
        [cleaned_question]
    )


    # Calculate cosine similarity
    similarity_scores = cosine_similarity(
        user_vector,
        faq_vectors
    )[0]


    # Find highest similarity
    best_index = similarity_scores.argmax()


    # Get highest score
    best_score = similarity_scores[best_index]


    # Minimum similarity threshold
    threshold = 0.15


    # If no suitable FAQ is found
    if best_score < threshold:

        return (
            "🤖 Sorry, I couldn't find a suitable "
            "answer to your question.\n\n"
            "Please try asking your question in "
            "a different way."
        )


    # Get matching FAQ answer
    answer = faqs[best_index]["answer"]


    return answer


# ============================================================
# CLEAR CHAT
# ============================================================

def clear_chat():

    return None


# ============================================================
# GRADIO INTERFACE
# ============================================================

with gr.Blocks(
    title="CodeAlpha FAQ Chatbot"
) as app:


    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.Markdown(
        """
        # 🤖 CodeAlpha FAQ Chatbot

        ### AI-powered Frequently Asked Questions Chatbot

        Ask a question and the chatbot will find
        the most similar FAQ using NLP.
        """
    )


    gr.Markdown("---")


    # --------------------------------------------------------
    # CHATBOT
    # --------------------------------------------------------

    chatbot = gr.Chatbot(
        label="FAQ Assistant",
        height=450
    )


    # --------------------------------------------------------
    # USER INPUT
    # --------------------------------------------------------

    user_input = gr.Textbox(
        label="Ask your question",
        placeholder="Example: How many tasks do I need to complete?",
        lines=2
    )


    # --------------------------------------------------------
    # BUTTONS
    # --------------------------------------------------------

    with gr.Row():

        ask_button = gr.Button(
            "💬 Ask",
            variant="primary"
        )

        clear_button = gr.Button(
            "🗑️ Clear"
        )


    # --------------------------------------------------------
    # CHAT FUNCTION
    # --------------------------------------------------------

    def chat_with_bot(
        question,
        history
    ):

        if history is None:
            history = []

        if not question or not question.strip():

            return (
                history,
                ""
            )


        # Get answer
        answer = get_answer(
            question
        )


        # Add conversation
        history.append(
            {
                "role": "user",
                "content": question
            }
        )

        history.append(
            {
                "role": "assistant",
                "content": answer
            }
        )


        return (
            history,
            ""
        )


    # --------------------------------------------------------
    # ASK BUTTON
    # --------------------------------------------------------

    ask_button.click(
        fn=chat_with_bot,
        inputs=[
            user_input,
            chatbot
        ],
        outputs=[
            chatbot,
            user_input
        ]
    )


    # --------------------------------------------------------
    # ENTER KEY
    # --------------------------------------------------------

    user_input.submit(
        fn=chat_with_bot,
        inputs=[
            user_input,
            chatbot
        ],
        outputs=[
            chatbot,
            user_input
        ]
    )


    # --------------------------------------------------------
    # CLEAR BUTTON
    # --------------------------------------------------------

    clear_button.click(
        fn=clear_chat,
        inputs=[],
        outputs=chatbot
    )


    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.Markdown(
        """
        ---

        **CodeAlpha Artificial Intelligence Internship**

        **Task 2 — Chatbot for FAQs**
        """
    )


# ============================================================
# LAUNCH APPLICATION
# ============================================================

app.launch(
    share=True
)



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://af816fad97efc34ae4.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
